# Gestore delle Spese Domestiche

Applicazione Python per il monitoraggio delle transazioni economiche quotidiane.

## Funzionalita'
1. **Aggiungi una transazione** - registra data, descrizione e importo in un file CSV
2. **Report Mensile** - visualizza le spese raggruppate per anno e mese
3. **Top 10 Transazioni** - mostra le 10 spese di maggior importo

**Istruzioni**: eseguire le celle nell'ordine indicato. L'ultima cella avvia il menu interattivo.

## 1. Importazioni e Configurazione

Importazione delle librerie standard necessarie per la gestione dei file CSV e delle date.

In [ ]:
import csv
import os
from datetime import datetime

# Path to the CSV file used as the expense database
PERCORSO_CSV = "spese.csv"

# Column headers for the CSV file
INTESTAZIONI_CSV = ["data", "descrizione", "importo"]

## 2. Gestione del File CSV

Funzioni di utilita' per la creazione e la lettura del file CSV che contiene le transazioni.

In [ ]:
def inizializza_csv(percorso: str) -> None:
    """
    Initializes the CSV file if it does not exist.
    Creates the file with the column headers.

    Args:
        percorso (str): Path to the CSV file to initialize.
    """
    if not os.path.exists(percorso):
        try:
            with open(percorso, mode="w", newline="", encoding="utf-8") as file_csv:
                writer = csv.writer(file_csv)
                writer.writerow(INTESTAZIONI_CSV)
            print(f"File '{percorso}' creato con successo.")
        except Exception as e:
            print(f"Errore durante la creazione del file '{percorso}': {e}")


def leggi_transazioni(percorso: str) -> list:
    """
    Reads all transactions from the CSV file.

    Args:
        percorso (str): Path to the CSV file to read.

    Returns:
        list: List of dictionaries containing the transactions.
              Each dictionary has the keys 'data', 'descrizione', 'importo'.
    """
    transazioni = []
    try:
        with open(percorso, mode="r", newline="", encoding="utf-8") as file_csv:
            reader = csv.DictReader(file_csv)
            for riga in reader:
                # Convert amount to float for numeric operations
                riga["importo"] = float(riga["importo"])
                transazioni.append(riga)
    except Exception as e:
        print(f"Errore durante la lettura del file '{percorso}': {e}")
    return transazioni

## 3. Funzioni di Input

Funzioni di utilita' per la lettura e la validazione dei dati inseriti dall'utente.

In [ ]:
def leggi_data(istruzioni: str) -> str:
    """
    Reads and validates a date in DD/MM/YYYY format.

    Args:
        istruzioni (str): The prompt message to display to the user.

    Returns:
        str: The valid date string in DD/MM/YYYY format.
    """
    while True:
        try:
            valore = input(istruzioni).strip()
            # Validate date format before accepting
            datetime.strptime(valore, "%d/%m/%Y")
            return valore
        except ValueError:
            print("Formato data non valido. Utilizzare il formato GG/MM/AAAA (es. 18/05/2024).")


def leggi_float(istruzioni: str) -> float:
    """
    Reads and validates a positive float value from input.

    Args:
        istruzioni (str): The prompt message to display to the user.

    Returns:
        float: The positive float value entered by the user.
    """
    while True:
        try:
            valore = float(input(istruzioni).strip())
            if valore <= 0:
                print("L'importo deve essere un valore positivo.")
                continue
            return valore
        except ValueError:
            print("Valore non valido. Inserire un numero (es. 45.50).")


def leggi_stringa(istruzioni: str) -> str:
    """
    Reads a non-empty string from input.

    Args:
        istruzioni (str): The prompt message to display to the user.

    Returns:
        str: The non-empty string entered by the user.
    """
    while True:
        valore = input(istruzioni).strip()
        if valore:
            return valore
        print("Il campo non puo' essere vuoto.")

## 4. Operazioni Principali

Implementazione delle tre funzionalita' chiave dell'applicazione:
- Aggiunta di una transazione
- Generazione del report mensile
- Visualizzazione delle Top 10 transazioni

In [ ]:
def aggiungi_transazione(percorso: str) -> None:
    """
    Adds a new transaction to the CSV file.
    Prompts the user for the date, description, and amount of the expense.

    Args:
        percorso (str): Path to the CSV file where the transaction will be saved.
    """
    print("\n--- AGGIUNGI TRANSAZIONE ---")

    data = leggi_data("Data (GG/MM/AAAA): ")
    descrizione = leggi_stringa("Descrizione: ")
    importo = leggi_float("Importo: ")

    try:
        with open(percorso, mode="a", newline="", encoding="utf-8") as file_csv:
            writer = csv.writer(file_csv)
            writer.writerow([data, descrizione, importo])
        print(f"Transazione salvata: {data} {descrizione} {importo:.2f}")
    except Exception as e:
        print(f"Errore durante il salvataggio della transazione: {e}")

In [ ]:
def report_mensile(percorso: str) -> None:
    """
    Generates and displays the monthly expense report.
    Transactions are sorted by date and displayed with their year-month prefix.

    Output format:
        YYYY-MM amount

    Args:
        percorso (str): Path to the CSV file to read transactions from.
    """
    print("\n--- REPORT MENSILE ---")

    transazioni = leggi_transazioni(percorso)

    if not transazioni:
        print("Nessuna transazione registrata.")
        return

    # Parse dates to allow correct chronological sorting
    for t in transazioni:
        t["_data_dt"] = datetime.strptime(t["data"], "%d/%m/%Y")

    # Sort transactions by date in ascending order
    transazioni_ordinate = sorted(transazioni, key=lambda t: t["_data_dt"])

    # Print each transaction with its YYYY-MM prefix
    for t in transazioni_ordinate:
        mese = t["_data_dt"].strftime("%Y-%m")
        print(f"{mese} {t['importo']:.2f}")

In [ ]:
def top_10_transazioni(percorso: str) -> None:
    """
    Displays the 10 transactions with the highest amount.
    Transactions are sorted in descending order by amount.

    Output format:
        DD/MM/YYYY description amount

    Args:
        percorso (str): Path to the CSV file to read transactions from.
    """
    print("\n--- TOP 10 TRANSAZIONI ---")

    transazioni = leggi_transazioni(percorso)

    if not transazioni:
        print("Nessuna transazione registrata.")
        return

    # Sort transactions by amount in descending order and take the top 10
    top_10 = sorted(transazioni, key=lambda t: t["importo"], reverse=True)[:10]

    for transazione in top_10:
        print(f"{transazione['data']} {transazione['descrizione']} {transazione['importo']:.2f}")

## 5. Menu Interattivo

Avvia l'applicazione con il menu interattivo principale.
Selezionare un'opzione digitando il numero corrispondente e premendo Invio.

In [ ]:
def stampa_menu() -> None:
    """
    Prints the main application menu.
    """
    print("\n" + "=" * 40)
    print("   GESTORE SPESE DOMESTICHE")
    print("=" * 40)
    print("[1] Aggiungi una transazione")
    print("[2] Report Mensile")
    print("[3] Top 10 Transazioni")
    print("[0] Esci")
    print("=" * 40)


def gestisci_scelta(scelta: str, percorso: str) -> bool:
    """
    Handles the user's selection from the main menu.

    Args:
        scelta (str): The user's input choice.
        percorso (str): Path to the transactions CSV file.

    Returns:
        bool: False if the user chose to exit, True otherwise.
    """
    if scelta == "1":
        aggiungi_transazione(percorso)
    elif scelta == "2":
        report_mensile(percorso)
    elif scelta == "3":
        top_10_transazioni(percorso)
    elif scelta == "0":
        print("Bye!")
        return False
    else:
        print("Scelta non valida. Selezionare un'opzione tra 0 e 3.")
    return True


def avvia_applicazione(percorso: str = PERCORSO_CSV) -> None:
    """
    Starts the Household Expense Manager application.
    Initializes the CSV file and presents the interactive menu to the user.

    Args:
        percorso (str): Path to the CSV file. Default: 'spese.csv'.
    """
    inizializza_csv(percorso)

    # Main application loop: runs until the user selects exit
    attivo = True
    while attivo:
        stampa_menu()
        scelta = input("Seleziona un'opzione: ").strip()
        attivo = gestisci_scelta(scelta, percorso)


avvia_applicazione()

## 6. Test delle Funzionalita' (Opzionale)

Cella opzionale per testare l'applicazione con dati di esempio, senza passare per il menu interattivo.

In [ ]:
# Separate test file to avoid mixing sample data with real expenses
PERCORSO_TEST = "spese_test.csv"


def popola_dati_test(percorso: str) -> None:
    """
    Populates the CSV file with sample data to test the application features.

    Args:
        percorso (str): Path to the test CSV file.
    """
    # Sample transactions covering multiple months and years
    dati_esempio = [
        ["18/05/2024", "Spesa al supermercato", 87.50],
        ["20/05/2024", "Bolletta elettricita'", 124.00],
        ["25/05/2024", "Cena al ristorante", 45.00],
        ["01/06/2024", "Abbonamento palestra", 50.00],
        ["05/06/2024", "Carburante auto", 65.30],
        ["10/06/2024", "Farmacia", 23.80],
        ["15/06/2024", "Rata mutuo", 834.00],
        ["18/06/2023", "Rata mutuo", 1231.00],
        ["15/02/2021", "Acquisto scooter", 4323.00],
        ["03/03/2022", "Riparazione auto", 450.00],
        ["22/07/2023", "Vacanze estive", 1800.00],
        ["08/11/2022", "PC portatile", 999.00],
    ]

    with open(percorso, mode="w", newline="", encoding="utf-8") as file_csv:
        writer = csv.writer(file_csv)
        writer.writerow(INTESTAZIONI_CSV)
        writer.writerows(dati_esempio)
    print(f"{len(dati_esempio)} transazioni di esempio inserite in '{percorso}'.")


popola_dati_test(PERCORSO_TEST)
print()
report_mensile(PERCORSO_TEST)
print()
top_10_transazioni(PERCORSO_TEST)